# Classical NLP: N-gram Surprisal

This notebook computes **bigram** and **trigram** surprisal using Kneser-Ney smoothing.

- **Natural Stories** is the primary dataset for main analysis
- **Dundee** is the verification/replication dataset

In [1]:
import os
import re
import sys
import math
import subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def resolve_data_paths():
    """Resolve dataset locations for local and Kaggle environments."""
    kaggle_input = "/kaggle/input"

    if os.path.exists(kaggle_input):
        # Accept either:
        # 1) <dataset>/data/natural_stories and <dataset>/data/dundee
        # 2) <dataset>/natural_stories and <dataset>/dundee
        for root, dirs, _ in os.walk(kaggle_input):
            dirs_set = set(dirs)

            if "data" in dirs_set:
                data_root = os.path.join(root, "data")
                ns_candidate = os.path.join(data_root, "natural_stories")
                du_candidate = os.path.join(data_root, "dundee")
                if os.path.exists(ns_candidate) and os.path.exists(du_candidate):
                    return ns_candidate, du_candidate, "/kaggle/working/results", "kaggle"

            if "natural_stories" in dirs_set and "dundee" in dirs_set:
                ns_candidate = os.path.join(root, "natural_stories")
                du_candidate = os.path.join(root, "dundee")
                return ns_candidate, du_candidate, "/kaggle/working/results", "kaggle"

        raise FileNotFoundError(
            "Could not locate natural_stories and dundee folders under /kaggle/input. "
            "Expected either <dataset>/data/... or <dataset>/... layout."
        )

    # Local fallback
    base = os.path.dirname(os.getcwd())
    ns_path = os.path.join(base, "data", "natural_stories")
    du_path = os.path.join(base, "data", "dundee")
    results_path = os.path.join(base, "results")
    return ns_path, du_path, results_path, "local"

DATA_NS, DATA_DU, RESULTS, RUN_ENV = resolve_data_paths()
os.makedirs(RESULTS, exist_ok=True)

print("Environment:", RUN_ENV)
print("Natural Stories path:", DATA_NS)
print("Dundee path:", DATA_DU)
print("Results path:", RESULTS)

Environment: kaggle
Natural Stories path: /kaggle/input/datasets/muddybuddy/computational-psycholinguistics/data/natural_stories
Dundee path: /kaggle/input/datasets/muddybuddy/computational-psycholinguistics/data/dundee
Results path: /kaggle/working/results


In [2]:
# Ensure nltk is available in the active kernel
try:
    import nltk
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])
    import nltk

for pkg in ["brown", "punkt"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import brown
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import KneserNeyInterpolated

print("NLTK ready. Brown sentences:", len(brown.sents()))

NLTK ready. Brown sentences: 57340


In [3]:
# Train bigram and trigram Kneser-Ney models on Brown corpus
train_sents = [[w.lower() for w in sent] for sent in brown.sents()]

# Bigram
n2 = 2
train_2, vocab_2 = padded_everygram_pipeline(n2, train_sents)
lm2 = KneserNeyInterpolated(order=n2)
lm2.fit(train_2, vocab_2)

# Trigram
n3 = 3
train_3, vocab_3 = padded_everygram_pipeline(n3, train_sents)
lm3 = KneserNeyInterpolated(order=n3)
lm3.fit(train_3, vocab_3)

print("Trained bigram and trigram Kneser-Ney models.")

Trained bigram and trigram Kneser-Ney models.


In [4]:
def normalize_token(token):
    if pd.isna(token):
        return "<unk>"
    token = str(token).strip().lower()
    # Keep letters, apostrophes, and hyphens; strip other punctuation for stable matching
    token = re.sub(r"[^a-z0-9'\-]+", "", token)
    return token if token else "<unk>"

def surprisal_bits(prob, eps=1e-12):
    p = max(float(prob), eps)
    return -math.log2(p)

def compute_group_surprisal(df, token_col, pos_col):
    seq = df.sort_values(pos_col).copy()
    seq["token"] = seq[token_col].map(normalize_token)

    history = []
    bi_vals = []
    tri_vals = []

    for tok in seq["token"].tolist():
        ctx2 = history[-1:]
        ctx3 = history[-2:]

        p2 = lm2.score(tok, ctx2)
        p3 = lm3.score(tok, ctx3)

        bi_vals.append(surprisal_bits(p2))
        tri_vals.append(surprisal_bits(p3))

        history.append(tok)

    seq["bigram_surprisal"] = bi_vals
    seq["trigram_surprisal"] = tri_vals
    return seq

# Persist displayed outputs so runs are reproducible in local/Kaggle
OUTPUTS_DIR = os.path.join(RESULTS, "notebook_outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

def save_display_df(df, filename):
    out_path = os.path.join(OUTPUTS_DIR, filename)
    df.to_csv(out_path, index=False)
    print("Saved display output:", out_path)
    return out_path

def save_text(content, filename):
    out_path = os.path.join(OUTPUTS_DIR, filename)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Saved text output:", out_path)
    return out_path

In [5]:
# Natural Stories surprisal
ns_words = pd.read_csv(os.path.join(DATA_NS, "ns_words.csv"))

ns_parts = []
for story_id, grp in tqdm(ns_words.groupby("story"), desc="Natural Stories"):
    out = compute_group_surprisal(grp, token_col="word_text", pos_col="zone")
    ns_parts.append(out)

ns_surp = pd.concat(ns_parts, ignore_index=True)
ns_out = os.path.join(RESULTS, "ns_ngram_surprisal.csv")
ns_surp[["story", "zone", "word_text", "token", "bigram_surprisal", "trigram_surprisal"]].to_csv(ns_out, index=False)

ns_head = ns_surp.head(10).copy()
ns_desc = ns_surp[["bigram_surprisal", "trigram_surprisal"]].describe().reset_index()
save_display_df(ns_head, "ns_surp_head10.csv")
save_display_df(ns_desc, "ns_surp_describe.csv")

save_text(
    f"Saved: {ns_out}\nRows: {len(ns_surp)}\n",
    "ns_run_log.txt"
)

print("Saved:", ns_out)
print("Rows:", len(ns_surp))
display(ns_head)
display(ns_desc)


Natural Stories:   0%|          | 0/10 [00:00<?, ?it/s]

Saved display output: /kaggle/working/results/notebook_outputs/ns_surp_head10.csv
Saved display output: /kaggle/working/results/notebook_outputs/ns_surp_describe.csv
Saved text output: /kaggle/working/results/notebook_outputs/ns_run_log.txt
Saved: /kaggle/working/results/ns_ngram_surprisal.csv
Rows: 10256


,story,zone,word_text,token,bigram_surprisal,trigram_surprisal
0,1,1,If,if,10.104788,10.104794
1,1,2,you,you,3.275795,4.337341
2,1,3,were,were,5.726070,5.235687
3,1,4,to,to,5.422809,5.962780
4,1,5,journey,journey,14.795042,18.714258
5,1,6,to,to,3.260074,6.529534
6,1,7,the,the,2.906802,0.647721
7,1,8,North,north,9.844486,7.960384
8,1,9,of,of,3.980828,8.364130
9,1,10,"England ,",england,10.631953,14.932945


,index,bigram_surprisal,trigram_surprisal
0,count,10256.000000,10256.000000
1,mean,11.599228,12.621784
2,std,8.513975,8.955463
3,min,0.036294,0.003154
4,25%,5.525545,5.798729
5,50%,9.943766,11.215880
6,75%,15.669897,17.332028
7,max,39.863137,39.863137


In [6]:
# Dundee surprisal
du_words = pd.read_csv(os.path.join(DATA_DU, "dundee_word_agg.csv"))

du_parts = []
for text_id, grp in tqdm(du_words.groupby("text_num"), desc="Dundee"):
    out = compute_group_surprisal(grp, token_col="word_text", pos_col="WNUM")
    du_parts.append(out)

du_surp = pd.concat(du_parts, ignore_index=True)
du_out = os.path.join(RESULTS, "dundee_ngram_surprisal.csv")
du_surp[["text_num", "WNUM", "word_text", "token", "bigram_surprisal", "trigram_surprisal"]].to_csv(du_out, index=False)

du_head = du_surp.head(10).copy()
du_desc = du_surp[["bigram_surprisal", "trigram_surprisal"]].describe().reset_index()
save_display_df(du_head, "du_surp_head10.csv")
save_display_df(du_desc, "du_surp_describe.csv")

save_text(
    f"Saved: {du_out}\nRows: {len(du_surp)}\n",
    "du_run_log.txt"
)

print("Saved:", du_out)
print("Rows:", len(du_surp))
display(du_head)
display(du_desc)


Dundee:   0%|          | 0/20 [00:00<?, ?it/s]

Saved display output: /kaggle/working/results/notebook_outputs/du_surp_head10.csv
Saved display output: /kaggle/working/results/notebook_outputs/du_surp_describe.csv
Saved text output: /kaggle/working/results/notebook_outputs/du_run_log.txt
Saved: /kaggle/working/results/dundee_ngram_surprisal.csv
Rows: 50650


,text_num,WNUM,word_text,mean_FDUR,mean_log_FDUR,mean_log_FDUR_z,n_subjects,word_len,line_num,wpos_in_line,word_freq_class,token,bigram_surprisal,trigram_surprisal
0,1,1,Are,154.285714,5.006451,-0.732279,7,3.0,1.0,1.0,351.0,are,8.162541,8.162547
1,1,2,tourists,173.600000,5.122610,-0.367107,10,8.0,2.0,2.0,3.0,tourists,20.289751,20.026067
2,1,3,enticed,227.200000,5.350754,0.298543,10,7.0,3.0,3.0,1.0,enticed,39.863137,39.863137
3,1,4,by,207.285714,5.302052,0.109256,7,2.0,4.0,4.0,252.0,by,7.592225,7.592232
4,1,5,these,173.900000,5.140666,-0.237526,10,5.0,5.0,5.0,73.0,these,8.563494,8.405654
5,1,6,attractions,213.000000,5.304290,0.128312,10,11.0,6.0,6.0,2.0,attractions,19.985227,23.134624
6,1,7,threatening,217.800000,5.338313,0.221195,10,11.0,7.0,7.0,3.0,threatening,18.315459,18.315466
7,1,8,their,227.000000,5.408758,0.379172,10,5.0,8.0,8.0,225.0,their,13.440832,13.325362
8,1,9,very,245.571429,5.488953,0.620393,7,4.0,9.0,9.0,56.0,very,8.580978,9.264801
9,1,10,existence?,208.700000,5.309332,0.194975,10,9.0,10.0,10.0,4.0,existence,8.707808,11.986191


,index,bigram_surprisal,trigram_surprisal
0,count,50650.000000,50650.000000
1,mean,12.650829,13.608374
2,std,9.405406,9.701282
3,min,0.007186,0.000445
4,25%,5.909238,6.311705
5,50%,10.803173,12.224971
6,75%,16.963648,18.220406
7,max,39.863137,39.863137


In [7]:
# Correlation checks against human reading difficulty
ns_agg = pd.read_csv(os.path.join(DATA_NS, "ns_word_agg.csv"))
du_agg = pd.read_csv(os.path.join(DATA_DU, "dundee_word_agg.csv"))

ns_eval = ns_agg.merge(
    ns_surp[["story", "zone", "bigram_surprisal", "trigram_surprisal"]],
    on=["story", "zone"],
    how="inner"
)
du_eval = du_agg.merge(
    du_surp[["text_num", "WNUM", "bigram_surprisal", "trigram_surprisal"]],
    on=["text_num", "WNUM"],
    how="inner"
)

ns_corr_bi = ns_eval["mean_log_RT"].corr(ns_eval["bigram_surprisal"])
ns_corr_tri = ns_eval["mean_log_RT"].corr(ns_eval["trigram_surprisal"])
du_corr_bi = du_eval["mean_log_FDUR"].corr(du_eval["bigram_surprisal"])
du_corr_tri = du_eval["mean_log_FDUR"].corr(du_eval["trigram_surprisal"])

summary = pd.DataFrame({
    "Dataset": ["Natural Stories (Primary)", "Dundee (Replication)"],
    "Corr(logDV, bigram surprisal)": [ns_corr_bi, du_corr_bi],
    "Corr(logDV, trigram surprisal)": [ns_corr_tri, du_corr_tri],
})

summary_view = summary.round(4)
save_display_df(summary_view, "ngram_correlation_summary_display.csv")

display(summary_view)

# Save quick evaluation table
summary.to_csv(os.path.join(RESULTS, "ngram_correlation_summary.csv"), index=False)
print("Saved:", os.path.join(RESULTS, "ngram_correlation_summary.csv"))
save_text(
    summary_view.to_string(index=False) + "\n",
    "ngram_correlation_summary_display.txt"
)

Saved display output: /kaggle/working/results/notebook_outputs/ngram_correlation_summary_display.csv


,Dataset,"Corr(logDV, bigram surprisal)","Corr(logDV, trigram surprisal)"
0,Natural Stories (Primary),0.2149,0.1928
1,Dundee (Replication),0.1732,0.1715


Saved: /kaggle/working/results/ngram_correlation_summary.csv
Saved text output: /kaggle/working/results/notebook_outputs/ngram_correlation_summary_display.txt


'/kaggle/working/results/notebook_outputs/ngram_correlation_summary_display.txt'